# 00 · Set up your Foundry project

*Before we optimize routing, is our project wired up and ready?*

This capsule changes one routing lever at a time and measures the impact — none of
that works until we have a project, a model-router deployment, and a prompt agent to
route. This notebook is a **self-contained quickstart** for this capsule: we create
everything once, confirm it works, and never print anything sensitive (so it's safe
to commit the output).

```mermaid
flowchart LR
    P[Create project<br/>contoso-travel-releases] --> M[Deploy models<br/>model-router · baselines · judge]
    M --> Ag[Create prompt agent<br/>contoso-travel-concierge]
    Ag --> E[.env: endpoint · deployment · agent]
    E --> V{Smoke test passes?}
    V -->|yes| Next[Proceed to 01]
    V -->|no| Fix[Fix and re-run]
```

## 1. Create the project

We keep this capsule's work in its own project so it's easy to find and tear down.

1. Go to [ai.azure.com](https://ai.azure.com) and sign in.
2. Create a new project named **`contoso-travel-releases`** (use the **Create
   project** wizard; complete the resource-group step if prompted).
3. From the project's overview, copy the **project endpoint**. It looks like
   `https://<resource>.services.ai.azure.com/api/projects/contoso-travel-releases`.

The shared [quickstart](../../quickstart/README.md) has the full walkthrough if you
want more detail. Make sure you've run `az login` in a terminal — this capsule
authenticates with your Azure identity, not an API key.

## 2. Deploy the models this capsule needs

In the project, open **Model catalog → Deploy** and name each deployment by the job
it does. Only the router is required to start; the rest unlock later notebooks.

| Deployment | Why | Needed by |
|---|---|---|
| `model-router` | The endpoint we route everything through | Every notebook (start here) |
| `cheap-baseline` (we use `gpt-5.4-nano`) | Cheap baseline for cost comparisons | 09 router-vs-fixed |
| `quality-baseline` (we use `gpt-5.4`) | Quality baseline, and the frontier model for the optimizer | 01 baseline, 09 router-vs-fixed |
| `scoring-judge` (we use `gpt-5.4`) | Scores candidate answers against the rubric | 01 baseline, optimizer notebooks |

### 2.1 Compare before you pick

Not sure which model fits a job? Use **Compare models** in the catalog to weigh
quality, safety, estimated cost, throughput, and context side by side.

For the **cheap baseline**, comparing `gpt-4.1-mini`, `gpt-5.4-mini`, and
`gpt-5.4-nano`, nano was cheapest and fastest (estimated cost 8.78, throughput 177)
while holding competitive quality — so we deploy nano.

![Compare on cost: mini vs nano](images/00-setup-compare-cost.png)

For the **quality baseline**, comparing `gpt-5.4-nano`, `gpt-5.4-mini`, and the base
`gpt-5.4`, the base model led on quality (0.81) — so we reach for it when quality is
the priority.

![Compare on quality: nano vs mini vs base](images/00-setup-compare-quality.png)

**Tip: name deployments by job, not by model.** We deployed nano as
**`cheap-baseline`** and gpt-5.4 as **`quality-baseline`**. A descriptive name lets
us swap the model behind it later without touching a single notebook or `.env` line.

### 2.2 Pick a scoring judge

The optimizer needs a model to **score** candidate answers. Comparing `gpt-5.6-sol`,
`gpt-5.4`, and `gpt-5.6-luna`, gpt-5.4 struck the best balance of quality and safety
at a reasonable cost — so we deploy it as **`scoring-judge`**.

![Compare judges: sol vs 5.4 vs luna](images/00-setup-compare-judge.png)

### 2.3 Deploy the router

Choose the **model-router** model and give it a capacity that fits your quota (this
project uses a 125K TPM deployment):

![Deploy the model-router](images/00-setup-router.png)

Notice the **Upgrade model version once a new default becomes available** checkbox is
on by default — so the router updates in place as new versions ship. Turn it off to
pin a specific version when you need a stable baseline to compare against.

The default routing pool spans the current supported models, so we don't pick an
underlying model — the router does, per request:

![model-router routing pool](images/00-setup-router-choices.png)

### 2.4 Create your local .env

We keep configuration in a **local `.env` in this folder** — not the shared quickstart
`.env` — so this capsule stays self-contained. From this folder, copy the template and
fill in your values:

```bash
cp sample.env .env    # then open .env and set the endpoint + deployment names
```

`.env` is git-ignored, so it never gets committed. The next cell loads it from this
folder.

In [16]:
## 3. Install packages and load .env
%pip install azure-ai-projects azure-identity openai python-dotenv --quiet

from dotenv import load_dotenv

# Load the .env you copied from sample.env in this capsule folder.
load_dotenv(".env")
print("Packages loaded and .env read from this folder (if present).")

Note: you may need to restart the kernel to use updated packages.
Packages loaded and .env read from this folder (if present).


## 4. Are the capsule variables set?

Three variables drive this capsule. We print only whether each is present, plus a
masked host — never the full endpoint or any secret.

In [17]:
## 4. Check variables (values stay masked)
import os
from urllib.parse import urlparse

REQUIRED = ["MICROSOFT_FOUNDRY_ENDPOINT", "AZURE_MODEL_ROUTER_DEPLOYMENT", "AGENT_NAME"]
status = {name: bool(os.getenv(name)) for name in REQUIRED}

def masked_host(url: str) -> str:
    # Reveal only the platform domain, never the resource or project names.
    host = urlparse(url).netloc
    suffix = ".".join(host.split(".")[-4:]) if host else "?"
    return f"https://***.{suffix}/api/projects/***"

for name, present in status.items():
    print(f"{name:<32} {'OK' if present else 'MISSING'}")

endpoint = os.getenv("MICROSOFT_FOUNDRY_ENDPOINT")
if endpoint:
    print("Endpoint (masked):", masked_host(endpoint))

if not all(status.values()):
    raise SystemExit("Set the missing variables (see the steps above) and re-run this cell.")

MICROSOFT_FOUNDRY_ENDPOINT       OK
AZURE_MODEL_ROUTER_DEPLOYMENT    OK
AGENT_NAME                       OK
Endpoint (masked): https://***.services.ai.azure.com/api/projects/***


## 5. Can we authenticate with Entra ID?

With `az login` done, `DefaultAzureCredential` can fetch a token. We confirm it
works without printing the token.

In [18]:
## 5. Confirm authentication (no token is printed)
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
try:
    credential.get_token("https://ai.azure.com/.default")
    print("Authentication: Success (token acquired, not shown).")
except Exception as exc:
    print("Authentication failed:", type(exc).__name__)
    raise SystemExit("Run `az login` in a terminal, then re-run this cell.")

Authentication: Success (token acquired, not shown).


## 6. Create the Contoso Travel Concierge prompt agent

A prompt agent bundles a model with instructions. We point its model at the
`model-router` deployment — so the router picks the underlying model per request —
and load the instructions from the shared [Contoso Travel demo](../../../demos/contoso-travel/).
We start from the grounded rung; later notebooks swap in higher rungs to show the
climb. Re-running is safe: if the agent already exists, we reuse it instead of
creating another version.

To fully ground the agent, attach the demo's `data/*.json` files as **Knowledge** on
the agent in the portal (Agents → your agent → Knowledge). The smoke test below
works without that step; grounding sharpens the answers.

In [19]:
## 6. Create the prompt agent (idempotent — reuse if it already exists)
from pathlib import Path
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

project = AIProjectClient(endpoint=endpoint, credential=credential)
agent_name = os.environ["AGENT_NAME"]

try:
    agent = project.agents.get(agent_name)
    print(f"Agent '{agent_name}' already exists (version {agent.version}) — reusing it.")
except Exception:
    instr_path = Path("../../../demos/contoso-travel/instructions/01-grounding.md")
    raw = instr_path.read_text(encoding="utf-8")
    # Drop the authoring comment header; keep only the instruction body.
    instructions = raw.split("-->", 1)[-1].strip() if raw.lstrip().startswith("<!--") else raw
    agent = project.agents.create_version(
        agent_name=agent_name,
        definition=PromptAgentDefinition(
            model=os.environ["AZURE_MODEL_ROUTER_DEPLOYMENT"],
            instructions=instructions,
        ),
    )
    print(f"Agent '{agent_name}' created (version {agent.version}).")

Agent 'contoso-travel-concierge' created (version 1).


## 7. Smoke test the agent

One neutral request confirms the whole path works end to end — and shows which
underlying model the router chose.

In [20]:
## 7. Send one request through the agent
import time

from IPython.display import Markdown, display


def ask(prompt, *, agent=None, model=None):
    """Send one request (via the agent or a router deployment) and pretty-print
    the reply plus the router's signals: chosen model, latency, and tokens."""
    client = project.get_openai_client(agent_name=agent) if agent else project.get_openai_client()
    kwargs = {"input": prompt}
    if agent:
        kwargs["conversation"] = client.conversations.create().id
    if model:
        kwargs["model"] = model

    start = time.perf_counter()
    resp = client.responses.create(**kwargs)
    latency_ms = round((time.perf_counter() - start) * 1000)

    usage = getattr(resp, "usage", None)
    tokens = getattr(usage, "total_tokens", None) if usage else None
    signals = f"**Model:** `{getattr(resp, 'model', 'unknown')}`  |  **Latency:** {latency_ms} ms"
    if tokens is not None:
        signals += f"  |  **Tokens:** {tokens} (drive cost)"
    display(Markdown(f"{resp.output_text or '_(no text output)_'}\n\n---\n{signals}"))
    return resp



In [21]:


# Smoke test: one neutral request confirms the whole path and shows the signals.
ask("In one sentence, what can you help me with?", agent=os.environ["AGENT_NAME"]);

I can help you find and compare flights, hotels, and car rentals from the available options, and assist with planning your work travel.

---
**Model:** `gpt-5.4-mini-2026-03-17`  |  **Latency:** 1901 ms  |  **Tokens:** 299 (drive cost)

## 8. Your Turn

Setup is a one-time gate — make it yours. The next few cells let you feel grounding
and routing before we make them measurable in notebook 01.

### 8.1 Upload the grounding files

Attach the demo's data as **Knowledge** so the agent answers from it:

1. In the portal, open **Agents → contoso-travel-concierge → Knowledge → Add**.
2. Upload the JSON files from [`demos/contoso-travel/data/`](../../../demos/contoso-travel/data/):
   `flights.json`, `hotels.json`, `car-rentals.json`, `travel-policy.json`
   (add `employees.json` for preference questions).
3. Save. New conversations now search these files.

![Add the data files as Knowledge](images/00-setup-knowledge.png)

### 8.2 Try asking an ungrounded question

Ask for something the catalog doesn't contain. A well-grounded agent should say it
can't find a match rather than invent one.

**Look for:** a clear "no matching flight" — not a made-up itinerary.

In [22]:
## 8.2 Ungrounded question — expect a decline
ask("Find me a nonstop flight from Boston to Sydney next month.", 
    agent=os.environ["AGENT_NAME"]);

I couldn’t find any nonstop flights from Boston (BOS) to Sydney (SYD) in the available inventory for next month, September 2026.

---
**Model:** `gpt-5.6-luna-2026-07-09`  |  **Latency:** 7790 ms  |  **Tokens:** 5825 (drive cost)

### 8.3 Try asking a grounded question

Now ask something the catalog can answer. A grounded agent should cite real option
ids and apply policy.

**Look for:** `F001` (ANA, $5,830, compliant) and `F002` (United, $6,300) flagged as
over the $6,000 cap — no invented flights.

In [23]:
## 8.3 Grounded question — expect real options + policy
ask(
    "Which nonstop business-class flights are available from Boston to Tokyo, and are they within policy?",
    agent=os.environ["AGENT_NAME"],
);

Available nonstop business-class flights from Boston to Tokyo (NRT):

| ID | Carrier / flight | Departure | Total fare | Policy status |
|---|---|---:|---:|---|
| **F001** | ANA NH9 | Nov 12, 2026, 10:30 | **$5,830** | **Within policy** |
| **F002** | United UA79 | Nov 12, 2026, 11:00 | **$6,300** | **Not within policy as-is** — exceeds the $6,000 international business-class cap by $300. It is eligible for a VP exception because the overspend is within the permitted 15%. |
| **F013** | ANA NH9 | Sep 1, 2026, 10:30 | **$5,780** | **Not within policy as-is** — booked only 11 days in advance, inside the required 14-day advance-purchase window. A manager waiver is required for urgent business travel. |

All three are refundable, nonstop, and have scheduled flight times over six hours, so business class is permitted. **F001 is the only option fully within policy without an exception.**

---
**Model:** `gpt-5.6-luna-2026-07-09`  |  **Latency:** 15538 ms  |  **Tokens:** 6396 (drive cost)

### 8.4 Try the cost router

Deploy a second router in **Cost** mode named `model-router-cost`, add it to `.env` as
`AZURE_MODEL_ROUTER_COST_DEPLOYMENT`, then send the same grounded question straight to
it. Calling the router directly isolates the routing choice, so it skips the agent's
grounding — we're watching the model pick, not the answer.

**Look for:** a cheaper underlying model and fewer tokens than Balanced.

In [24]:
## 8.4 Cost router — same question, cheaper pick
query = "Which nonstop business-class flights are available from Boston to Tokyo, and are they within policy?"
cost_router = os.getenv("AZURE_MODEL_ROUTER_COST_DEPLOYMENT")

if not cost_router:
    print("Set AZURE_MODEL_ROUTER_COST_DEPLOYMENT in .env after deploying a Cost-mode router.")
else:
    ask(query, model=cost_router)

I can help, but I need two quick clarifications to answer “within policy” correctly:

1) **What dates (or date range)** are you traveling? Nonstop Japan routes from Boston are often **seasonal**.  
2) What does your **travel policy** allow/limit (e.g., must be **nonstop only**, allowed cabin is **Business**, max total travel time, whether **NRT vs HND** is preferred, any “must be lowest available fare” rule)?

## What nonstop business-class options commonly exist (BOS → Tokyo)
> Note: I can’t see live schedules/inventory from here, so consider this a **likely set of nonstop products** you should verify for your exact dates.

### 1) **ANA (All Nippon Airways)** — **BOS ⇄ HND (Tokyo Haneda)**
- Typically **operated nonstop** between Boston and Tokyo Haneda.
- **Business class** is available (flat-bed seats, varies by aircraft).

### 2) **United Airlines** — **BOS ⇄ NRT (Tokyo Narita)**
- United has operated **nonstop** BOS–Tokyo Narita service at times.
- **Business class** available (United Polaris).

### 3) **JAL (Japan Airlines)** — may operate **BOS ⇄ HND** (route can be seasonal)
- Japan Airlines has previously offered nonstop BOS–Tokyo Haneda on certain periods/dates.
- **Business class** available (JAL business products).

## Are they “within policy”?
If your policy generally permits:
- **Business class**, and
- **Nonstop flights** (or “avoid connections where reasonable”),
then **any of the above nonstop BOS → Tokyo options** would usually be considered **within policy**.

However, if your policy includes rules like:
- “Only the lowest available fare,”
- “Allow business only if the discounted economy/cabin isn’t feasible,”
- “Max flight time” or specific airport preferences,
then I’d need your policy text (or the exact constraints) to confirm.

---

### Next step
Reply with:
- **Travel date(s)** (or approximate month),
- Your **policy rules** (paste the relevant section),
- Whether you prefer **HND or NRT** (or “either is fine”).

…and I’ll tell you **which nonstop business-class flights are actually available for those dates** and whether they **comply with your policy**.

---
**Model:** `gpt-5.4-nano-2026-03-17`  |  **Latency:** 13659 ms  |  **Tokens:** 1178 (drive cost)

### 8.5 Try the quality router

Do the same with a **Quality**-mode router named `model-router-quality`
(`AZURE_MODEL_ROUTER_QUALITY_DEPLOYMENT`).

**Look for:** a stronger underlying model — usually higher quality, more tokens, and a
little more latency than Cost.

In [25]:
## 8.5 Quality router — same question, stronger pick
quality_router = os.getenv("AZURE_MODEL_ROUTER_QUALITY_DEPLOYMENT")

if not quality_router:
    print("Set AZURE_MODEL_ROUTER_QUALITY_DEPLOYMENT in .env after deploying a Quality-mode router.")
else:
    ask(query, model=quality_router)

The only nonstop Boston–Tokyo business-class service is:

| Option | Operating airline | Typical flight | Route | Cabin | Policy status |
|---|---|---:|---|---|---|
| Japan Airlines nonstop | Japan Airlines | JL 7 | Boston Logan — Tokyo Narita | Business Class | Likely within policy if business class is allowed for long-haul international travel |
| Codeshare of same flight | Often sold by partners such as American Airlines | e.g., AA codeshare on JL 7 | BOS — NRT | Business Class | Same physical flight; policy treatment depends on fare and preferred-carrier rules |

There is generally no nonstop Boston–Tokyo Haneda flight. The nonstop service is to Tokyo Narita.

Policy assessment: the flight is about 13–14 hours nonstop and international/transpacific, so under most travel policies that allow business class for flights over 6–8 hours, it would be within policy. If your company policy requires lowest logical fare, preferred airline, fare cap, or advance-purchase approval, I’d need the travel date and quoted fare to confirm fully.

---
**Model:** `gpt-5.5-2026-04-24`  |  **Latency:** 18573 ms  |  **Tokens:** 1295 (drive cost)

### 8.6 Try the balanced router

Balanced is the default mode of the `model-router` deployment we already have
(`AZURE_MODEL_ROUTER_DEPLOYMENT`). Send the same question one more time so the Cost,
Quality, and Balanced picks sit back to back.

**Look for:** a middle-ground pick — between Cost's cheaper model and Quality's stronger one.

In [26]:
## 8.6 Balanced router — same question, the default trade-off
# Balanced is the default mode of the model-router deployment we already have.
ask(query, model=os.environ["AZURE_MODEL_ROUTER_DEPLOYMENT"]);

I’ll need your travel date(s) and your company’s travel-policy rules to confirm real-time availability and compliance.

Boston (BOS) typically has these nonstop Tokyo options in business class:

| Route | Carrier | Tokyo airport | Typical cabin |
|---|---|---:|---|
| BOS → NRT | Japan Airlines (JAL) | Narita | Business Class |
| BOS → HND | Delta Air Lines | Haneda | Delta One |

Whether either is within policy depends on items such as:

- permitted cabin by trip length or traveler level;
- maximum fare or lowest-logical-fare requirement;
- preferred carriers / contracted airlines;
- required advance-purchase window;
- whether Haneda versus Narita is an approved airport.

Send the departure date, one-way or round-trip details, preferred Tokyo airport, and the applicable policy (or fare cap), and I can assess the options.

---
**Model:** `gpt-5.6-terra-2026-07-09`  |  **Latency:** 6869 ms  |  **Tokens:** 470 (drive cost)

## 9. Summary

We created the `contoso-travel-releases` project, deployed a `model-router`, stood up
the `contoso-travel-concierge` prompt agent from the shared demo instructions, and
confirmed the path end to end — without printing anything sensitive.

Then we felt the two ideas this capsule is built on. **Grounding**: with the data
attached, the agent cited real options and declined the Boston–Sydney request instead
of inventing one. **Routing**: sending the *same* question to the Balanced, Cost, and
Quality routers, we watched the signals shift — Cost leaned to a cheaper, faster model
with fewer tokens; Quality leaned to a stronger model at a little more latency and
cost; Balanced sat between. That quality–cost–latency trade-off is the whole game.

Eyeballing one question tells a story, but it isn't a decision. Next, notebook 01 turns
these signals into a **scorecard** across the Contoso benchmark, and we bring in
**evaluations** to score quality properly rather than by eye.

See the [capsule overview](README.md) for the full series, and the
[glossary](../../../docs/GLOSSARY.md) if a term is new.

## 10. References

- [Create a prompt agent](https://learn.microsoft.com/en-us/azure/foundry/agents/quickstarts/prompt-agent?tabs=python) — the `AIProjectClient` + `PromptAgentDefinition` pattern used here.
- [Create a project in the Foundry portal](https://learn.microsoft.com/en-us/azure/foundry/how-to/create-projects)
- [Deploy models in Microsoft Foundry](https://learn.microsoft.com/en-us/azure/foundry/how-to/deploy-models-openai)
- [Model Releases quickstart](../../quickstart/README.md) · [Contoso Travel demo](../../../demos/contoso-travel/)